# LAB 4b:  Keras DNN モデルの作成

**学習目標**

1. CSV 列、ラベル列、および列のデフォルト値を設定する
1. CSV ファイルから特徴量とラベルのデータセットを作成する
1. 生の特徴量の入力レイヤを作成する
1. DNN の Dense 隠れレイヤと出力レイヤを作成する
1. カスタム評価指標を作成する
1. すべてのパーツを統合して DNN モデルを構築する
1. トレーニングと評価を行う


## はじめに 
このノートブックでは、Keras を使用して、生まれる前の赤ちゃんの体重を予測する DNN モデルを作成します。

まず、データ入力用の CSV 列名、ラベル列、および列のデフォルト値を定義します。次に、CSV ファイルから特徴量とラベルの `tf.data Dataset` を構築し、生の特徴量の入力レイヤを作成します。続いて、モデル入力用の前処理レイヤを設定し、Keras でディープ ニューラル ネットワークを構築します。カスタム評価指標を作成して DNN モデルを構築します。最後に、モデルのトレーニングと評価を行います。

各学習目標は、この受講者用ラボ ノートブック内の __#TODO__ に対応しています。まずこのノートブックを完了してみてから、[解答ノートブック](../solutions/4b_keras_dnn_babyweight.ipynb) を確認してください。

## 必要なライブラリの読み込み

In [ ]:
import datetime
import os
import shutil

import keras
import matplotlib.pyplot as plt
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## CSV ファイルが存在することの確認

このシリーズの 7 番目のラボ [4a_sample_babyweight](../solutions/4a_sample_babyweight.ipynb) では、BigQuery からトレーニング用、評価用、およびテスト用の CSV ファイルをサンプリングしました。それらのファイルが存在することを確認し、存在しない場合はそのラボに戻って作成してください。

In [ ]:
%%bash
ls *.csv

In [ ]:
%%bash
head -5 *.csv

## Keras モデルの作成

### CSV 列、ラベル列、および列のデフォルト値の設定

CSV ファイルが存在することを確認したので、入力関数で使用するいくつかの設定を行う必要があります。
* `CSV_COLUMNS` は列のヘッダー名になります。CSV ファイルと同じ順序になっていることを確認してください
* `LABEL_COLUMN` はラベルである列のヘッダー名です。特徴量辞書からポップするためにこれを知る必要があります。
* `DEFAULTS` は `CSV_COLUMNS` と同じ長さのリストです（つまり、CSV の各列にデフォルト値があります）。各要素自体が、その CSV 列のデフォルト値を含むリストになります。

In [ ]:
# CSV、ラベル、キー列を決定
# 文字列の列ヘッダーのリストを作成し、順序が一致していることを確認します。
CSV_COLUMNS = [
    "weight_pounds",
    "is_male",
    "mother_age",
    "plurality",
    "gestation_weeks",
]

# ラベル列の文字列名を追加
LABEL_COLUMN = "weight_pounds"

NUMERICAL_COLUMNS = ["mother_age", "gestation_weeks"]
CATEGORICAL_COLUMNS = ["is_male", "plurality"]

# 各 CSV 列のデフォルト値をリストのリストとして設定します。
# is_male と plurality を文字列として扱います。
DEFAULTS = [[0.0], ["null"], [0.0], ["null"], [0.0]]

### CSV ファイルからの特徴量とラベルのデータセットの作成

次に、データを読み込むための `input_fn` を記述します。これにより、CSV データセット オブジェクトが作成されます。ただし、列を特徴量とラベルに分割する必要があります。これを行うには、データセットに map メソッドを適用し、特徴量テンソルの辞書からラベル列をポップします。

In [ ]:
def features_and_labels(row_data):
    """Splits features and labels from feature dictionary.

    Args:
        row_data: Dictionary of CSV column names and tensor values.
    Returns:
        Dictionary of feature tensors and label tensor.
    """
    label = row_data.pop(LABEL_COLUMN)

    return row_data, label


def decode_csv(line):
    fields = tf.io.decode_csv(line, record_defaults=DEFAULTS)
    return dict(zip(CSV_COLUMNS, fields))


def load_dataset(pattern, batch_size=1, mode="eval"):
    """Loads dataset using the tf.data API from CSV files.

    Args:
        pattern: str, file pattern to glob into list of files.
        batch_size: int, the number of examples per batch.
        mode: called when training or evaluating
    Returns:
        `Dataset` object.
    """
    dataset = tf.data.Dataset.list_files(pattern, shuffle=(mode == "train"))

    # TextLineDataset を使用してこれらのファイルの内容を読み取ります
    # interleave により複数のファイルから並列に読み取ることができます
    dataset = dataset.interleave(
        tf.data.TextLineDataset, num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.map(decode_csv, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(
        map_func=features_and_labels, num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(batch_size)
    # トレーニング用にシャッフル
    if mode == "train":
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.repeat().prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset

### 生の特徴量の入力レイヤの作成

入力関数によって読み込まれたデータをモデル関数に渡す必要がありますが、どのように連携させればよいでしょうか？以下を定義することで Keras 入力レイヤを使用できます:
* shape: バッチサイズを含まない shape タプル（整数）。たとえば、shape=(32,) は想定される入力が 32 次元ベクトルのバッチであることを示します。このタプルの要素は None にすることができます。「None」要素は shape が不明な次元を表します。
* name: レイヤのオプションの名前文字列。モデル内で一意である必要があります（同じ名前を 2 回再利用しないでください）。指定しない場合は自動生成されます。
* dtype: 入力で想定されるデータ型を文字列（float32、float64、int32 など）で指定します。

In [ ]:
def create_input_layers():
    """Creates dictionary of input layers for each feature.

    Returns:
        Dictionary of `keras.Input` layers for each feature.
    """
    inputs = {
        colname: keras.Input(name=colname, shape=(1,), dtype="float32")
        for colname in NUMERICAL_COLUMNS
    }

    inputs.update(
        {
            colname: keras.Input(name=colname, shape=(1,), dtype="string")
            for colname in CATEGORICAL_COLUMNS
        }
    )

    return inputs

### 入力用の前処理レイヤの定義

次に、生の特徴量を変換するための前処理レイヤを定義します。`mother_age` と `gestation_weeks` は数値にする必要があります。他の `is_male` と `plurality` はカテゴリカルにする必要があります。

In [ ]:
def transform(inputs):
    """Creates dictionary of transformed inputs.

    Returns:
        Dictionary of transformed Tensors
    """

    transformed = {}

    for numerical_column in NUMERICAL_COLUMNS:
        transformed[numerical_column] = inputs[numerical_column]

    vocab = {
        "is_male": ["True", "False", "Unknown"],
        "plurality": [
            "Single(1)",
            "Twins(2)",
            "Triplets(3)",
            "Quadruplets(4)",
            "Quintuplets(5)",
            "Multiple(2+)",
        ],
    }

    for categorical_column in CATEGORICAL_COLUMNS:
        transformed[categorical_column] = keras.layers.StringLookup(
            vocabulary=vocab[categorical_column], output_mode="one_hot"
        )(inputs[categorical_column])

    return transformed

### DNN の Dense 隠れレイヤと出力レイヤの作成

機械学習用に入力を準備する方法を理解したので、次はそれらを目的の出力に接続する必要があります。モデル アーキテクチャがこれら 2 つを結びつけます。入力から始まるいくつかの隠れ Dense レイヤを作成し、Dense 出力レイヤで終わるようにします。これは回帰問題であるため、出力レイヤの活性化関数と shape が正しいことを確認してください。

In [ ]:
def get_model_outputs(inputs):
    """Creates model architecture and returns outputs.

    Args:
        inputs: Dense tensor used as inputs to model.
    Returns:
        Dense tensor output from the model.
    """
    # [64, 32] の 2 つの隠れレイヤを作成
    h1 = keras.layers.Dense(64, activation="relu", name="h1")(inputs)
    h2 = keras.layers.Dense(32, activation="relu", name="h2")(h1)

    # これは回帰であるため、最終出力は線形活性化（linear activation）です
    output = keras.layers.Dense(units=1, activation="linear", name="weight")(h2)

    return output

### すべてのパーツを統合した DNN モデルの構築

素晴らしい！すべてのパーツが揃ったので、それらを 1 つの Keras Model にまとめるだけです。これは分岐やサイド入力などのないシンプルなフィードフォワード モデルであるため、Keras の Sequential Model API を使用することもできましたが、ここでは練習のために Keras の Functional Model API を使用します。ここでは `keras.models.Model` を使用して入力と出力を指定してモデルを構築し、オプティマイザ、損失関数、および評価指標を使用してモデルをコンパイルします。

In [ ]:
def build_dnn_model():
    """Builds simple DNN using Keras Functional API.

    Returns:
        `keras.models.Model` object.
    """
    # 入力レイヤを作成
    inputs = create_input_layers()

    # 変換（transform）
    transformed = transform(inputs)

    # Keras の Functional API では LayerConstructor()(inputs) が必要です
    dnn_inputs = keras.layers.Concatenate()(list(transformed.values()))

    # 入力を与えてモデルの出力を取得
    output = get_model_outputs(dnn_inputs)

    # モデルを構築し、すべてまとめてコンパイル
    model = keras.models.Model(inputs=inputs, outputs=output)
    rmse = keras.metrics.RootMeanSquaredError(
        name="root_mean_squared_error", dtype=None
    )
    model.compile(optimizer="adam", loss="mse", metrics=[rmse, "mse"])

    return model


print("Here is our DNN architecture so far:\n")
model = build_dnn_model()
print(model.summary())

Keras の plot_model ユーティリティを使用して DNN を可視化できます。

In [ ]:
keras.utils.plot_model(model=model, show_layer_names=True, rankdir="LR")

## モデルの実行と評価

### トレーニングと評価

CSV ファイルからの入力と設計したアーキテクチャを使用して Keras モデルを構築しました。それでは、モデル パラメータをトレーニングし、トレーニングの進行に合わせて未知のデータに対して定期的に評価を実行して精度を追跡し、モデルを実行しましょう。トレーニング データセットと評価データセットの両方を読み込み、fit メソッドを通じてそれらをモデルに渡す必要があります。データを読み込むときは、正しいパターン、バッチサイズ、およびモードが設定されていることを確認してください。また、TensorBoard へのコールバックを追加することも忘れないでください。

In [ ]:
TRAIN_BATCH_SIZE = 32
NUM_TRAIN_EXAMPLES = 10000 * 5  # トレーニング データセットはリピートされ、折り返されます
NUM_EVALS = 5  # 評価を実行する回数
# 適切なサンプルを取得するのに十分であり、処理が遅くならない程度の量
NUM_EVAL_EXAMPLES = 10000

trainds = load_dataset(
    pattern="train*",
    batch_size=TRAIN_BATCH_SIZE,
    mode="train",
)

evalds = load_dataset(pattern="eval*", batch_size=1000, mode="eval").take(
    count=NUM_EVAL_EXAMPLES // 1000
)

steps_per_epoch = NUM_TRAIN_EXAMPLES // (TRAIN_BATCH_SIZE * NUM_EVALS)

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=logdir, histogram_freq=1
)

history = model.fit(
    trainds,
    validation_data=evalds,
    epochs=NUM_EVALS,
    steps_per_epoch=steps_per_epoch,
    callbacks=[tensorboard_callback],
)

### 損失曲線の可視化

In [ ]:
history.history

In [ ]:
# プロット
import matplotlib.pyplot as plt

nrows = 1
ncols = 2
fig = plt.figure(figsize=(10, 5))

for idx, key in enumerate(["loss", "root_mean_squared_error"]):
    ax = fig.add_subplot(nrows, ncols, idx + 1)
    plt.plot(history.history[key])
    plt.plot(history.history[f"val_{key}"])
    plt.title(f"model {key}")
    plt.ylabel(key)
    plt.xlabel("epoch")
    plt.legend(["train", "validation"], loc="upper left");

### モデルの保存

In [ ]:
OUTPUT_DIR = "babyweight_trained"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(OUTPUT_DIR, TIMESTAMP)

SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")
KERAS_PATH = os.path.join(EXPORT_PATH, "model.keras")

model.export(EXPORT_PATH)
model.save(KERAS_PATH)

In [ ]:
!ls $EXPORT_PATH

## ラボのまとめ: 
このラボでは、まずデータ入力用の CSV 列名、ラベル列、および列のデフォルト値を定義しました。次に、CSV ファイルから特徴量とラベルの tf.data Dataset を構築し、生の特徴量の入力レイヤを作成しました。続いて、モデル入力用の前処理レイヤを設定し、Keras でディープ ニューラル ネットワークを構築しました。カスタム評価指標を作成して DNN モデルを構築しました。最後に、モデルのトレーニングと評価を行いました。

Copyright 2026 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.